# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.1.0 | Voyce | 13 Jul 2026 |
| v0.2.0 | Voyce | 16 Sep 2026 |


In [ ]:
# Import supported public objects from the root package.
# Make sure the Fabric environment already has FabricOps installed as a custom library.

from fabricops_kit import (
    DataAgreementConfig,
    FabricStore,
    FrameworkConfig,
    GovernanceConfig,
    PathConfig,
    setup_metadata_tables,
    setup_notebook,
)

## Path config

Define the environment and the logical Fabric stores available to downstream notebooks.
Logical store names are project-configurable. FabricOps reserves `metadata` for the metadata Lakehouse.


In [ ]:
# Select the current engineering runtime environment.
# ENV chooses which configured path set engineering notebooks use; it does not define
# the Governance environment. Add dev, qat, prod, or any organization-specific
# environments as separate ENV_PATHS entries.
ENV = "dev"

# One physical Metadata Lakehouse is shared by every engineering environment.
# Each environment still creates its own FabricStore below so store.env preserves
# the engineering environment that produced runtime metadata.
METADATA_WORKSPACE_ID = "68fa4319-1945-458f-bd21-05334c51cbb4"
METADATA_ITEM_ID = "329b3989-4546-4331-b9ff-df898d49ee73"

ENV_PATHS = {
    "dev": {
        "Bronze": FabricStore(
            env="dev",
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="ed3aad28-de5d-43d5-8a97-a6988901c921",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Silver": FabricStore(
            env="dev",
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="107d4b73-7c0e-4ce1-8da8-7602ac4a1372",
            kind="lakehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Gold": FabricStore(
            env="dev",
            workspace_id="68fa4319-1945-458f-bd21-05334c51cbb4",
            item_id="185346e5-6ce4-40c8-852a-185f1a933d20",
            kind="warehouse",
            schema_enabled=True,
            schema="dbo",
        ),
        "Metadata": FabricStore(
            env="dev",
            workspace_id=METADATA_WORKSPACE_ID,
            item_id=METADATA_ITEM_ID,
            kind="lakehouse",
            schema_enabled=True,
            # Store-level default only. Canonical metadata writers route tables to
            # FrameworkConfig governance and engineering schemas by table ownership.
            schema="governance",
        ),
    },

    # Add further engineering environments as peer entries when needed.
    # Keep each FabricStore.env aligned with its ENV_PATHS key. Every environment
    # reuse METADATA_WORKSPACE_ID + METADATA_ITEM_ID for "Metadata" so Governance has
    # one shared control plane while engineering records retain env provenance.
    #
    # "prod": {
    #     "Bronze": FabricStore(env="prod", workspace_id="...", item_id="...", kind="lakehouse", ...),
    #     "Silver": FabricStore(env="prod", workspace_id="...", item_id="...", kind="lakehouse", ...),
    #     "Gold": FabricStore(env="prod", workspace_id="...", item_id="...", kind="warehouse", ...),
    #     "Metadata": FabricStore(
    #         env="prod",
    #         workspace_id=METADATA_WORKSPACE_ID,
    #         item_id=METADATA_ITEM_ID,
    #         kind="lakehouse",
    #         schema_enabled=True,
    #         schema="governance",
    #     ),
    # },
}

PATH_CONFIG = PathConfig(paths=ENV_PATHS)

## 01_governance metadata intake config

The Steward and Agreement widgets in `01_governance` expose only lightweight business fields. Add organization-specific fields here; the widgets store those values in `custom_fields_json` without changing package code or table schemas.


In [ ]:
DATA_AGREEMENT_CONFIG = DataAgreementConfig(
    metadata_tables={
        "data_steward": "METADATA_DATA_STEWARD",
        "data_agreement": "METADATA_DATA_AGREEMENT",
    },
    steward_role_options=[
        "Data Owner",
        "Data Steward",
        "Data Custodian",
        "Governance Reviewer",
        "Business Approver",
    ],
    data_steward_widget={
        "visible_columns": [
            "steward_name", "steward_role", "contact", "effective_from", "effective_to",
        ],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "dropdown",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
    data_agreement_widget={
        "visible_columns": [
            "agreement_name", "domain", "provider_steward_id", "recipient_steward_id",
            "recipient", "start_date", "expiry_date", "business_purpose",
        ],
        "approved_usage_options": ["internal cross domain", "internal single domain", "research", "external"],
        "custom_fields": [
            {
                "key": "optional",
                "label": "Optional",
                "type": "text",
                "required": False,
                "help": "Optional group of users or organization unit covered by the agreement.",
            },
            {
                "key": "optional_dropdown",
                "label": "Optional dropdown",
                "type": "dropdown",
                "required": False,
                "options": ["ODI", "Faculty", "Department", "Research Group", "Other"],
            },
        ],
    },
)

## 01_governance enrichment config

Configure the governed information-classification labels and the optional Fabric AI suggestions used by the Data Contract editor. Classification itself is manual: users choose from `sensitivity_labels`; FabricOps does not ask AI to classify the table or column.

When AI is enabled, FabricOps builds the relevant metadata/profile context internally, serializes it into one prompt, places that prompt in a one-row pandas DataFrame, and calls Fabric AI Functions. The notebook user does not pass a DataFrame to the AI helper. Description uses table/column context, Sensitive Data assesses the selected column, and Data Quality proposes standard rules for the selected column. Suggestions are review-only until the user explicitly accepts or saves them.


In [ ]:
GOVERNANCE_CONFIG = GovernanceConfig(
    sensitivity_labels=["Public", "Internal", "Confidential", "Restricted"],
    ai_enrichment={
        "enabled": True,
        # Description AI receives one selected scope at a time. For a table, FabricOps supplies the
        # table Catalogue identity and existing description. For a column, it also supplies datatype
        # and profile summary evidence. Classification is intentionally NOT AI-generated.
        "description_prompt": """Write a concise business description for the selected table or column.

FabricOps will append a structured Context object to this prompt. For a table, that context contains the table-level Catalogue metadata and current description. For a column, it also contains the selected column name, datatype, and profile-summary evidence such as row, null, distinct, minimum, and maximum statistics when available.

Use only that supplied evidence. Profile statistics describe what was observed; they do not prove business meaning. Do not invent owners, processes, definitions, relationships, or intended usage that are not supported by the context. Return only the proposed description.""",
        # Sensitive Data AI is scoped to the selected column. It sees table context plus the selected
        # column's metadata, current Description/Classification, profile summary, and governed frequency
        # evidence. Classification remains a human-selected input signal, not an AI decision.
        "sensitive_data_prompt": """Suggest an advisory Sensitive Data Guardrail for the selected active canonical Catalogue column.

FabricOps will append table context and one selected column with its name, datatype, current description, manually selected information classification, profile-summary evidence, and available governed frequency evidence. Assess that column as Direct PII, Indirect PII, or Not PII.

Briefly explain the evidence for the assessment. For Direct or Indirect PII, suggest only Tokenize, Mask, Bucket, or Remove with explicit parameters and a Warn or Block action. Treat the manually selected Classification as context only; it is not itself proof of PII. Never request additional raw rows or return raw values. Return a structured JSON list containing exactly the supplied column; final review belongs to Governance.""",
        # Data Quality AI is also scoped to the selected column. It proposes only standard structured
        # rule families that can be reviewed and edited in the widget before anything is saved.
        "dq_prompt": """Suggest conservative standard Data Quality rules for the selected column.

FabricOps will append the selected column's governed metadata, current description, manually selected classification, profile-summary statistics, and available frequency evidence. Use only Completeness, Uniqueness, Value Set, Range, and Pattern.

Observed values are evidence, not an automatic contract: zero nulls does not prove a field is required, current distinctness does not prove uniqueness, observed categories do not automatically define an allowed set, and observed minima or maxima do not automatically define contractual bounds. Never suggest cross-column relationships, custom expressions, SQL, Python, or executable code. Return structured JSON only; final review belongs to Governance.""",
    },
)

## Config compiler and bootstrap

Assemble the shared config once. Downstream functions resolve store paths and canonical metadata schemas from this config rather than duplicating those choices in notebook code.


In [ ]:
# FabricOps audit and Spark session timezone.
# UTC is the portable default. Use a valid IANA timezone when local audit time is required.
FABRICOPS_AUDIT_TIMEZONE = "Asia/Singapore"

CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    data_agreement_config=DATA_AGREEMENT_CONFIG,
    audit_timezone=FABRICOPS_AUDIT_TIMEZONE,
)

# Keep native Spark timestamps and render/parse them using the validated audit timezone.
spark.conf.set("spark.sql.session.timeZone", CONFIG.audit_timezone)

# Validate every logical store declared for this environment.
RUN_CONTEXT = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=list(CONFIG.path_config.paths[ENV]),
)

In [ ]:
# Expose the minimal shared context consumed by downstream FabricOps resolvers.
# Store identities and metadata schemas are resolved from CONFIG when needed.
import builtins

FABRIC_CONTEXT = {
    "env": ENV,
    "config": CONFIG,
    "runtime_metadata": RUN_CONTEXT.runtime_metadata,
}
builtins.FABRIC_CONTEXT = FABRIC_CONTEXT

print(f"Active Fabric context initialized for environment: {ENV}")

In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")

for store_name, store in CONFIG.path_config.paths[ENV].items():
    print(f"- {store_name}")

## Metadata table setup

Create or validate the canonical FabricOps metadata tables in the configured `metadata` Lakehouse.
FabricOps resolves each table to its owning `governance` or `engineering` schema from `FrameworkConfig`; this notebook does not assign one schema to all metadata tables.


In [ ]:
METADATA_TABLE_SETUP = setup_metadata_tables(
    spark=spark,
    config=CONFIG,
    env=ENV,
    require_active_steward=False,
)

In [ ]:
print("FabricOps environment ready")
print(f"- audit timezone: {CONFIG.audit_timezone}")
print(f"- Spark session timezone: {spark.conf.get('spark.sql.session.timeZone')}")
print(f"- governance metadata schema: {CONFIG.governance_metadata_schema}")
print(f"- engineering metadata schema: {CONFIG.engineering_metadata_schema}")
print(f"- active metadata tables: {METADATA_TABLE_SETUP['active_metadata_table_count']}")